# Markdown

# 09 - Final Model & Prediction Module

## Objective

Create a reusable budget expenditure prediction pipeline for the healthcare budget ML project.

This notebook:

1. Loads the engineered budget dataset.
2. Validates the required prediction features.
3. Recreates the preprocessing used in Notebook 05.
4. Trains Linear Regression, Random Forest, and XGBoost pipelines.
5. Uses the model comparison from Notebook 08 as the documented basis for the selected model.
6. Fits the selected pipeline on all available complete historical rows.
7. Saves the final model as a reusable `.joblib` artifact.
8. Saves model metadata and prediction results.
9. Provides a reusable `predict_budget()` function for later API integration.

### Important limitation

The current budget dataset contains only a small number of annual periods. Therefore, the saved model is a project prototype/inference artifact and should not be described as a validated long-term forecasting system.

Notebook 05 showed that Linear Regression had the lowest MAE and RMSE and the highest R² on its chronological test set. Notebook 09 therefore preserves that documented selection while still training and retaining XGBoost as a project-required model.

# Imports

In [16]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


# Find project root

In [17]:
working_directory = Path.cwd().resolve()

project_root = next(
    (
        candidate
        for candidate in (working_directory, *working_directory.parents)
        if (candidate / "notebooks").is_dir()
        and (candidate / "data" / "processed").is_dir()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        f"Could not locate project root from {working_directory}"
    )

print("Project root:", project_root)

Project root: C:\Users\bagad\health-budget-ml


# Define paths

In [18]:
processed_data_directory = (
    project_root / "data" / "processed"
)

models_directory = (
    project_root / "models"
)

predictions_directory = (
    project_root / "outputs" / "predictions"
)

reports_directory = (
    project_root / "outputs" / "reports"
)

for directory in [
    models_directory,
    predictions_directory,
    reports_directory,
]:
    directory.mkdir(parents=True, exist_ok=True)

budget_data_path = (
    processed_data_directory / "budget_features.csv"
)

final_model_path = (
    models_directory / "budget_model.joblib"
)

xgboost_model_path = (
    models_directory / "budget_xgboost_model.joblib"
)

metadata_path = (
    models_directory / "budget_model_metadata.json"
)

final_predictions_path = (
    predictions_directory / "final_budget_predictions.csv"
)

print("Budget dataset:", budget_data_path)
print("Final model:", final_model_path)
print("XGBoost model:", xgboost_model_path)
print("Metadata:", metadata_path)

Budget dataset: C:\Users\bagad\health-budget-ml\data\processed\budget_features.csv
Final model: C:\Users\bagad\health-budget-ml\models\budget_model.joblib
XGBoost model: C:\Users\bagad\health-budget-ml\models\budget_xgboost_model.joblib
Metadata: C:\Users\bagad\health-budget-ml\models\budget_model_metadata.json


# Load budget data

In [19]:
if not budget_data_path.exists():
    raise FileNotFoundError(
        f"Budget feature dataset not found: {budget_data_path}"
    )

budget = pd.read_csv(budget_data_path)

print("Dataset shape:", budget.shape)
display(budget.head())

Dataset shape: (108, 8)


,State,Year,Year_Numeric,Central_Release,Expenditure_to_Release_Pct,Previous_Year_Expenditure,Previous_Year_Central_Release,Expenditure
0,Andaman and Nicobar Islands,2021-22,2021,43.68,71.428571,NaN,NaN,31.20
1,Andaman and Nicobar Islands,2022-23,2022,45.26,93.415820,31.20,43.68,42.28
2,Andaman and Nicobar Islands,2023-24,2023,37.84,107.584567,42.28,45.26,40.71
3,Andhra Pradesh,2021-22,2021,1199.37,204.163019,NaN,NaN,2448.67
4,Andhra Pradesh,2022-23,2022,1489.45,128.526637,2448.67,1199.37,1914.34


# Validate required columns

In [20]:
target_column = "Expenditure"

predictor_columns = [
    "State",
    "Year_Numeric",
    "Central_Release",
    "Previous_Year_Expenditure",
    "Previous_Year_Central_Release",
]

required_columns = predictor_columns + [target_column]

missing_columns = sorted(
    set(required_columns) - set(budget.columns)
)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("All required columns are present.")

print("\nPredictors:")
for column in predictor_columns:
    print(" -", column)

print("\nTarget:", target_column)

All required columns are present.

Predictors:
 - State
 - Year_Numeric
 - Central_Release
 - Previous_Year_Expenditure
 - Previous_Year_Central_Release

Target: Expenditure


# Leakage validation

In [21]:
forbidden_predictors = [
    "Expenditure",
    "Expenditure_to_Release_Pct",
]

unexpected_predictors = [
    column
    for column in predictor_columns
    if column in forbidden_predictors
]

if unexpected_predictors:
    raise ValueError(
        f"Potential target leakage detected: {unexpected_predictors}"
    )

print("Leakage check passed.")
print("Current-year expenditure is not used as a predictor.")
print("Expenditure_to_Release_Pct is not used as a predictor.")

Leakage check passed.
Current-year expenditure is not used as a predictor.
Expenditure_to_Release_Pct is not used as a predictor.


# Prepare complete modeling data

In [22]:
model_data = budget[
    predictor_columns + [target_column]
].copy()

print("Rows before missing-value filtering:", len(model_data))

missing_counts = model_data.isna().sum()

display(
    missing_counts.to_frame("Missing_Count")
)

model_data = model_data.dropna(
    subset=predictor_columns + [target_column]
).copy()

print(
    "Rows after removing incomplete modeling rows:",
    len(model_data)
)

if len(model_data) == 0:
    raise ValueError("No complete rows are available for model training.")

Rows before missing-value filtering: 108


,Missing_Count
State,0
Year_Numeric,0
Central_Release,0
Previous_Year_Expenditure,36
Previous_Year_Central_Release,36
Expenditure,0


Rows after removing incomplete modeling rows: 72


# Data type validation

In [23]:
model_data["State"] = model_data["State"].astype(str)

numeric_features = [
    "Year_Numeric",
    "Central_Release",
    "Previous_Year_Expenditure",
    "Previous_Year_Central_Release",
]

for column in numeric_features:
    model_data[column] = pd.to_numeric(
        model_data[column],
        errors="coerce"
    )

model_data = model_data.dropna(
    subset=numeric_features + [target_column]
).copy()

print("Final training rows:", len(model_data))
print("\nData types:")
display(model_data.dtypes)

Final training rows: 72

Data types:


State                                str
Year_Numeric                       int64
Central_Release                  float64
Previous_Year_Expenditure        float64
Previous_Year_Central_Release    float64
Expenditure                      float64
dtype: object

# Show available years

In [24]:
available_years = sorted(
    model_data["Year_Numeric"]
    .astype(int)
    .unique()
    .tolist()
)

print("Complete years available for final model:")
print(available_years)

print("\nNumber of complete years:", len(available_years))

if len(available_years) < 3:
    print(
        "\nWARNING: The dataset has fewer than 3 complete annual periods."
    )
    print(
        "The resulting model should be treated as a prototype inference artifact."
    )

Complete years available for final model:
[2022, 2023]

Number of complete years: 2

The resulting model should be treated as a prototype inference artifact.


# Define preprocessing

In [25]:
categorical_features = [
    "State"
]

def make_preprocessor(scale_numeric=False):
    numeric_transformer = (
        StandardScaler()
        if scale_numeric
        else "passthrough"
    )

    return ColumnTransformer(
        transformers=[
            (
                "state",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                categorical_features,
            ),
            (
                "numeric",
                numeric_transformer,
                numeric_features,
            ),
        ],
        remainder="drop",
    )

print("Preprocessing configuration created.")

Preprocessing configuration created.


# Define models

In [26]:
models = {
    "Linear Regression": Pipeline(
        steps=[
            (
                "preprocessor",
                make_preprocessor(scale_numeric=True)
            ),
            (
                "model",
                LinearRegression()
            ),
        ]
    ),

    "Random Forest": Pipeline(
        steps=[
            (
                "preprocessor",
                make_preprocessor(scale_numeric=False)
            ),
            (
                "model",
                RandomForestRegressor(
                    n_estimators=150,
                    max_depth=3,
                    min_samples_leaf=2,
                    random_state=42,
                    n_jobs=1,
                )
            ),
        ]
    ),

    "XGBoost": Pipeline(
        steps=[
            (
                "preprocessor",
                make_preprocessor(scale_numeric=False)
            ),
            (
                "model",
                XGBRegressor(
                    objective="reg:squarederror",
                    n_estimators=60,
                    random_state=42,
                    n_jobs=1,
                )
            ),
        ]
    ),
}

print("Models created:")
for model_name in models:
    print(" -", model_name)

Models created:
 - Linear Regression
 - Random Forest
 - XGBoost


# Prepare X and y

In [27]:
X = model_data[predictor_columns].copy()
y = model_data[target_column].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nPredictors:")
print(list(X.columns))

print("\nTarget:")
print(target_column)

X shape: (72, 5)
y shape: (72,)

Predictors:
['State', 'Year_Numeric', 'Central_Release', 'Previous_Year_Expenditure', 'Previous_Year_Central_Release']

Target:
Expenditure


# Train all three models

In [28]:
trained_models = {}

for model_name, pipeline in models.items():
    print(f"Training {model_name}...")

    pipeline.fit(X, y)

    trained_models[model_name] = pipeline

    print(f"{model_name} training complete.")

print("\nAll models trained successfully.")

Training Linear Regression...
Linear Regression training complete.
Training Random Forest...
Random Forest training complete.
Training XGBoost...
XGBoost training complete.

All models trained successfully.


# Generate full-data predictions

In [29]:
prediction_results = model_data[
    ["State", "Year_Numeric", target_column]
].copy()

for model_name, pipeline in trained_models.items():
    prediction_column = (
        model_name
        .lower()
        .replace(" ", "_")
        .replace("-", "")
        + "_prediction"
    )

    prediction_results[prediction_column] = (
        pipeline.predict(X)
    )

display(prediction_results.head(10))

,State,Year_Numeric,Expenditure,linear_regression_prediction,random_forest_prediction,xgboost_prediction
1,Andaman and Nicobar Islands,2022,42.28,31.604720,111.331555,42.525005
2,Andaman and Nicobar Islands,2023,40.71,51.385280,111.331555,40.472839
4,Andhra Pradesh,2022,1914.34,2163.369783,2682.013150,1916.258301
5,Andhra Pradesh,2023,2436.62,2187.590217,2523.271375,2435.442871
7,Arunachal Pradesh,2022,424.78,349.474116,295.029160,424.172363
8,Arunachal Pradesh,2023,399.44,474.745884,359.663818,399.270416
10,Assam,2022,2307.00,2368.522643,2971.737612,2307.963623
11,Assam,2023,2626.87,2565.347357,3293.000619,2626.606689
13,Bihar,2022,3717.81,3930.240236,2944.690778,3717.372070
14,Bihar,2023,4010.97,3798.539764,4164.134742,4011.510254


# Save final model

In [36]:
# Cell 16: Load budget data and prepare final models

from pathlib import Path
import pandas as pd
import numpy as np
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor


# ---------------------------------------------------------
# 1. Find project root
# ---------------------------------------------------------

current_path = Path.cwd()

if (current_path / "data" / "processed").exists():
    project_root = current_path
elif (current_path.parent / "data" / "processed").exists():
    project_root = current_path.parent
else:
    project_root = current_path

print("Project root:", project_root)


# ---------------------------------------------------------
# 2. Define directories
# ---------------------------------------------------------

processed_data_directory = project_root / "data" / "processed"
models_directory = project_root / "models"
predictions_directory = project_root / "outputs" / "predictions"

models_directory.mkdir(parents=True, exist_ok=True)
predictions_directory.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------
# 3. Define file paths
# ---------------------------------------------------------

budget_data_path = (
    processed_data_directory / "budget_features.csv"
)

final_model_path = (
    models_directory / "budget_model.joblib"
)

xgboost_model_path = (
    models_directory / "budget_xgboost_model.joblib"
)

final_predictions_path = (
    predictions_directory / "final_budget_predictions.csv"
)


# ---------------------------------------------------------
# 4. Load budget dataset
# ---------------------------------------------------------

print("\nLoading:", budget_data_path)

budget_data = pd.read_csv(budget_data_path)

print("Dataset shape:", budget_data.shape)
display(budget_data.head())


# ---------------------------------------------------------
# 5. Define target and predictors
# ---------------------------------------------------------

target_column = "Expenditure"

predictor_columns = [
    "State",
    "Year_Numeric",
    "Central_Release",
    "Previous_Year_Expenditure",
    "Previous_Year_Central_Release"
]


# ---------------------------------------------------------
# 6. Check required columns
# ---------------------------------------------------------

required_columns = predictor_columns + [target_column]

missing_columns = [
    col for col in required_columns
    if col not in budget_data.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("\nAll required columns are present.")


# ---------------------------------------------------------
# 7. Prepare data
# ---------------------------------------------------------

budget_data["State"] = budget_data["State"].astype(str)

numeric_columns = [
    "Year_Numeric",
    "Central_Release",
    "Previous_Year_Expenditure",
    "Previous_Year_Central_Release",
    "Expenditure"
]

for col in numeric_columns:
    budget_data[col] = pd.to_numeric(
        budget_data[col],
        errors="coerce"
    )

budget_data = budget_data.dropna(
    subset=required_columns
).copy()

print("Complete rows available:", len(budget_data))


# ---------------------------------------------------------
# 8. Separate X and y
# ---------------------------------------------------------

X_full = budget_data[predictor_columns].copy()
y_full = budget_data[target_column].copy()


# ---------------------------------------------------------
# 9. Preprocessing
# ---------------------------------------------------------

categorical_features = ["State"]

numeric_features = [
    "Year_Numeric",
    "Central_Release",
    "Previous_Year_Expenditure",
    "Previous_Year_Central_Release"
]


def make_preprocessor(scale_numeric=False):

    if scale_numeric:
        numeric_transformer = StandardScaler()
    else:
        numeric_transformer = "passthrough"

    return ColumnTransformer(
        transformers=[
            (
                "state",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                categorical_features
            ),
            (
                "numeric",
                numeric_transformer,
                numeric_features
            )
        ],
        remainder="drop"
    )


# ---------------------------------------------------------
# 10. Define models
# ---------------------------------------------------------

models = {

    "Linear Regression": Pipeline([
        (
            "preprocessor",
            make_preprocessor(scale_numeric=True)
        ),
        (
            "model",
            LinearRegression()
        )
    ]),

    "Random Forest": Pipeline([
        (
            "preprocessor",
            make_preprocessor(scale_numeric=False)
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=150,
                max_depth=3,
                min_samples_leaf=2,
                random_state=42,
                n_jobs=1
            )
        )
    ]),

    "XGBoost": Pipeline([
        (
            "preprocessor",
            make_preprocessor(scale_numeric=False)
        ),
        (
            "model",
            XGBRegressor(
                objective="reg:squarederror",
                n_estimators=60,
                random_state=42,
                n_jobs=1
            )
        )
    ])
}


# ---------------------------------------------------------
# 11. Train all final models
# ---------------------------------------------------------

final_models = {}

for model_name, model in models.items():

    print(f"Training {model_name}...")

    model.fit(X_full, y_full)

    final_models[model_name] = model

print("\nAll final models trained successfully.")


# ---------------------------------------------------------
# 12. Select final deployment model
# ---------------------------------------------------------

selected_model_name = "Linear Regression"

selected_model = final_models[selected_model_name]

print("\nSelected deployment model:")
print(selected_model_name)

Project root: c:\Users\bagad\health-budget-ml

Loading: c:\Users\bagad\health-budget-ml\data\processed\budget_features.csv
Dataset shape: (108, 8)


,State,Year,Year_Numeric,Central_Release,Expenditure_to_Release_Pct,Previous_Year_Expenditure,Previous_Year_Central_Release,Expenditure
0,Andaman and Nicobar Islands,2021-22,2021,43.68,71.428571,NaN,NaN,31.20
1,Andaman and Nicobar Islands,2022-23,2022,45.26,93.415820,31.20,43.68,42.28
2,Andaman and Nicobar Islands,2023-24,2023,37.84,107.584567,42.28,45.26,40.71
3,Andhra Pradesh,2021-22,2021,1199.37,204.163019,NaN,NaN,2448.67
4,Andhra Pradesh,2022-23,2022,1489.45,128.526637,2448.67,1199.37,1914.34



All required columns are present.
Complete rows available: 72
Training Linear Regression...
Training Random Forest...
Training XGBoost...

All final models trained successfully.

Selected deployment model:
Linear Regression


In [37]:
# Cell 17: Save final selected model

joblib.dump(
    selected_model,
    final_model_path
)

print("Final model saved successfully:")
print(final_model_path)

Final model saved successfully:
c:\Users\bagad\health-budget-ml\models\budget_model.joblib


In [38]:
# Cell 18: Save XGBoost model

joblib.dump(
    final_models["XGBoost"],
    xgboost_model_path
)

print("XGBoost model saved successfully:")
print(xgboost_model_path)

XGBoost model saved successfully:
c:\Users\bagad\health-budget-ml\models\budget_xgboost_model.joblib


In [39]:
# Cell 19: Generate final predictions

final_prediction_data = budget_data[
    ["State", "Year_Numeric", "Expenditure"]
].copy()

for model_name, model in final_models.items():

    prediction_column = (
        model_name.lower()
        .replace(" ", "_")
        .replace("-", "_")
        + "_prediction"
    )

    final_prediction_data[prediction_column] = (
        model.predict(X_full)
    )

display(final_prediction_data.head(10))

,State,Year_Numeric,Expenditure,linear_regression_prediction,random_forest_prediction,xgboost_prediction
1,Andaman and Nicobar Islands,2022,42.28,31.604720,111.331555,42.525005
2,Andaman and Nicobar Islands,2023,40.71,51.385280,111.331555,40.472839
4,Andhra Pradesh,2022,1914.34,2163.369783,2682.013150,1916.258301
5,Andhra Pradesh,2023,2436.62,2187.590217,2523.271375,2435.442871
7,Arunachal Pradesh,2022,424.78,349.474116,295.029160,424.172363
8,Arunachal Pradesh,2023,399.44,474.745884,359.663818,399.270416
10,Assam,2022,2307.00,2368.522643,2971.737612,2307.963623
11,Assam,2023,2626.87,2565.347357,3293.000619,2626.606689
13,Bihar,2022,3717.81,3930.240236,2944.690778,3717.372070
14,Bihar,2023,4010.97,3798.539764,4164.134742,4011.510254


In [40]:
# Cell 20: Save final predictions

final_prediction_data.to_csv(
    final_predictions_path,
    index=False
)

print("Final predictions saved successfully:")
print(final_predictions_path)

Final predictions saved successfully:
c:\Users\bagad\health-budget-ml\outputs\predictions\final_budget_predictions.csv


In [41]:
# Cell 21: Reload the final model

loaded_model = joblib.load(final_model_path)

print("Final model loaded successfully.")
print(loaded_model)

Final model loaded successfully.
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('state',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['State']),
                                                 ('numeric', StandardScaler(),
                                                  ['Year_Numeric',
                                                   'Central_Release',
                                                   'Previous_Year_Expenditure',
                                                   'Previous_Year_Central_Release'])])),
                ('model', LinearRegression())])


In [42]:
# Cell 22: Reusable budget prediction function

def predict_budget(
    state,
    year_numeric,
    central_release,
    previous_year_expenditure,
    previous_year_central_release
):
    input_data = pd.DataFrame([{
        "State": state,
        "Year_Numeric": year_numeric,
        "Central_Release": central_release,
        "Previous_Year_Expenditure": previous_year_expenditure,
        "Previous_Year_Central_Release": previous_year_central_release
    }])

    prediction = loaded_model.predict(
        input_data[predictor_columns]
    )[0]

    return float(prediction)

print("Prediction function created successfully.")

Prediction function created successfully.


In [43]:
# Cell 23: Test prediction

test_row = budget_data.iloc[0]

test_prediction = predict_budget(
    state=test_row["State"],
    year_numeric=test_row["Year_Numeric"],
    central_release=test_row["Central_Release"],
    previous_year_expenditure=test_row["Previous_Year_Expenditure"],
    previous_year_central_release=test_row["Previous_Year_Central_Release"]
)

print("State:", test_row["State"])
print("Year:", test_row["Year_Numeric"])
print("Actual Expenditure:", test_row["Expenditure"])
print("Predicted Expenditure:", test_prediction)

State: Andaman and Nicobar Islands
Year: 2022
Actual Expenditure: 42.28
Predicted Expenditure: 31.604720168583526


In [44]:
# Cell 24: Validate prediction

if np.isfinite(test_prediction):
    print("Prediction validation: PASSED")
else:
    print("Prediction validation: FAILED")

Prediction validation: PASSED


In [45]:
# Cell 25: Verify saved files

required_files = [
    final_model_path,
    xgboost_model_path,
    final_predictions_path
]

print("File verification:\n")

for file_path in required_files:
    print(
        file_path,
        "->",
        "EXISTS" if file_path.exists() else "MISSING"
    )

File verification:

c:\Users\bagad\health-budget-ml\models\budget_model.joblib -> EXISTS
c:\Users\bagad\health-budget-ml\models\budget_xgboost_model.joblib -> EXISTS
c:\Users\bagad\health-budget-ml\outputs\predictions\final_budget_predictions.csv -> EXISTS


In [46]:
# Cell 26: Final status

all_files_exist = all(
    file_path.exists()
    for file_path in required_files
)

if all_files_exist and np.isfinite(test_prediction):

    print("======================================")
    print("FINAL MODEL MODULE: COMPLETED")
    print("======================================")
    print("Selected Model:", selected_model_name)
    print("XGBoost Model: Saved separately")
    print("Prediction Function: Ready")
    print("Model Artifact: Ready")
    print("Prediction Output: Ready")

else:

    print("Some validation checks failed.")

FINAL MODEL MODULE: COMPLETED
Selected Model: Linear Regression
XGBoost Model: Saved separately
Prediction Function: Ready
Model Artifact: Ready
Prediction Output: Ready
